[Ouvrir dans Colab](https://colab.research.google.com/github/moonlovist/EEEA-ML/blob/main/sessions/session02/lab/fr/02_Exercices_FR.ipynb)

# TP 02 · Exercices de programmation
Ting Tan · EEEA

De la régression à la classification.

## Prise en main
Exécuter les cellules dans l’ordre avec **Maj+Entrée**. Aucun GPU ni clé API n’est nécessaire. Les données sont incluses ; aucun fichier de données à importer.

**Colab :** ouvrir [Colab](https://colab.research.google.com/), choisir **Fichier → Importer un notebook**, importer ce `.ipynb` et se connecter à un environnement Python. Enregistrer/télécharger le notebook avant de quitter.

**Jupyter local :** dans un terminal du dossier extrait :
```
python -m pip install -r requirements.txt
python -m notebook
```
Si Jupyter est déjà configuré, utiliser l’environnement habituel.

Exécuter les imports ci-dessous. Si les résultats deviennent incohérents, redémarrer le noyau/l’environnement et tout exécuter dans l’ordre. Lire les erreurs à partir de leur dernière ligne.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score
)
np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 4), "font.size": 11})

def sigmoid(z):
    z = np.asarray(z, dtype=float)
    # Equivalent to 1 / (1 + exp(-z)), with stable evaluation.
    return np.exp(-np.logaddexp(0.0, -z))

def mean_bce(x, c, w, b):
    z = w * x + b
    return float(np.mean(np.logaddexp(0.0, z) - c * z))

print("Python / NumPy / Matplotlib / scikit-learn: OK")

Python / NumPy / Matplotlib / scikit-learn: OK


**Rappels Python :** `np.array` stocke un vecteur ; `*` multiplie terme à terme ; `np.mean` calcule une moyenne. `p >= 0.5` produit des booléens ; `np.sum(condition)` compte les valeurs `True`. Les indices commencent à 0.

`sigmoid` et `mean_bce` calculent de façon stable les formules du cours. Les logarithmes sont naturels.

## Travail à réaliser
Compléter **E1–E4** et leurs questions écrites. **B1–B3** approfondissent les mêmes notions.

Modifier vos fonctions et réponses sans changer les vérifications. `TODO` indique un code incomplet ; `OK` indique la réussite des vérifications. Ce notebook est indépendant des démonstrations.

Utiliser NumPy pour E1–E3 ; ne pas remplacer vos fonctions par un estimateur ou des mesures de bibliothèque.

## E1 · Une mise à jour linéaire réutilisable
Implémenter `linear_step(x, y, w, b, eta)`. x,y sont des tableaux NumPy 1D non vides de même longueur. Renvoyer `(w_new, b_new)`.

Calculer les résidus, puis les deux gradients **moyens** avec les anciens paramètres, puis la mise à jour simultanée.

Calculer le deuxième pas depuis $(w,b)=(10.2,0.1)$ avec $\eta=0.1$. Pourquoi modifier w avant le calcul du gradient de b donne-t-il un autre algorithme ?

In [2]:
def linear_step(x, y, w, b, eta):
    # TODO: compute predictions, residuals and both mean gradients.
    # TODO: return the two updated parameters.
    return None

In [3]:
x = np.array([1., 2., 3.])
y = np.array([12., 19., 32.])
result = linear_step(x, y, 10.2, 0.1, 0.1)
if result is None:
    print("TODO E1")
else:
    assert np.allclose(result, [10.286666666666667, 0.15])
    other = linear_step(np.array([0., 2.]), np.array([1., 5.]), 0., 0., 0.1)
    assert np.allclose(other, [.5, .3]), "Check the mean and simultaneous update."
    assert np.allclose(linear_step(x, y, 10.2, .1, 0.), [10.2, .1])
    print("E1: OK | second update =", result)

TODO E1


**Réponse :** _Écrire votre explication ici._

## E2 · Une fonction d’apprentissage logistique
Implémenter `fit_logistic(x, c, eta=0.1, updates=5000)` pour des tableaux 1D non vides de même longueur, avec des classes 0/1.

- Initialiser `w=b=0` ; effectuer exactement `updates` pas sur le lot complet.
- Renvoyer les clés `w`, `b`, `losses`.
- Enregistrer la perte initiale et après chaque pas : `len(losses) == updates + 1`.
- Utiliser `sigmoid` et `mean_bce`, déjà fournies.

Pourquoi `p-c` remplace-t-il le résidu numérique de E1 ? Que montre une courbe de perte décroissante, et que ne montre-t-elle pas ?

In [4]:
def fit_logistic(x, c, eta=0.1, updates=5000):
    w, b = 0.0, 0.0
    losses = [mean_bce(x, c, w, b)]
    # TODO: repeat prediction, gradient calculation and simultaneous update.
    # TODO: append the loss AFTER every update.
    # TODO: return {"w": w, "b": b, "losses": losses}.
    return None

In [5]:
x_binary = np.arange(1., 7.)
c_binary = np.array([0, 0, 1, 0, 1, 1])
model = fit_logistic(x_binary, c_binary)
if model is None:
    print("TODO E2")
else:
    assert len(model["losses"]) == 5001
    assert np.isclose(model["losses"][0], np.log(2))
    assert np.allclose([model["w"], model["b"]], [1.2134457869, -4.2468229786])
    assert np.isclose(model["losses"][-1], .4129978434013483)
    assert np.all(np.diff(model["losses"]) <= 1e-12)
    one = fit_logistic(np.array([0., 1.]), np.array([1, 0]), updates=1)
    assert np.allclose([one["w"], one["b"]], [-.025, 0.])
    zero = fit_logistic(x_binary, c_binary, updates=0)
    assert len(zero["losses"]) == 1 and zero["w"] == zero["b"] == 0.
    print("E2: OK | final BCE =", model["losses"][-1])
    plt.plot(model["losses"], color="#e43d26")
    plt.xlabel("Mise à jour")
    plt.ylabel("BCE moyenne")
    plt.tight_layout()
    plt.show()

TODO E2


**Réponse :** _Écrire votre explication ici._

## E3 · Un bilan de classification avec des masques booléens
Implémenter `classification_report_manual(y_true, probabilities, tau)` pour des tableaux NumPy 1D non vides de même longueur et des classes 0/1.

Utiliser $p\geq\tau$. Compter TP, FP, FN, TN, puis calculer exactitude, précision, rappel, F1 et spécificité. Renvoyer exactement les clés indiquées. Les effectifs sont entiers. Utiliser **0.0 si un dénominateur est nul**.

Comparer 0.5 et 0.8. Identifier les groupes formant les dénominateurs de la précision et du rappel. Pourquoi une même exactitude n’implique-t-elle pas des décisions équivalentes ?

In [6]:
def classification_report_manual(y_true, probabilities, tau):
    # TODO: predicted class is 1 when probability >= tau.
    # TODO: count TP, FP, FN, TN, then calculate the five metrics.
    # Return these exact keys:
    # TP, FP, FN, TN, accuracy, precision, recall, f1, specificity
    # Convention: return 0.0 for a metric whose denominator is zero.
    return None

In [7]:
truth = np.array([1, 1, 0, 1, 1, 0, 0, 0])
scores = np.array([.95, .85, .70, .60, .45, .30, .20, .10])
report = classification_report_manual(truth, scores, .5)
if report is None:
    print("TODO E3")
else:
    assert [report[k] for k in ["TP", "FP", "FN", "TN"]] == [3, 1, 1, 3]
    assert np.allclose([report[k] for k in
                       ["accuracy", "precision", "recall", "f1", "specificity"]],
                      [.75] * 5)
    high = classification_report_manual(truth, scores, .8)
    assert [high[k] for k in ["TP", "FP", "FN", "TN"]] == [2, 0, 2, 4]
    assert np.allclose([high["precision"], high["recall"], high["f1"]], [1, .5, 2/3])
    edge = classification_report_manual(np.array([1, 0]), np.array([.5, .5]), .5)
    assert edge["TP"] == edge["FP"] == 1, "Use >=, not >."
    empty_positive = classification_report_manual(np.array([0, 0]),
                                                 np.array([.1, .2]), .5)
    assert empty_positive["accuracy"] == 1 and empty_positive["precision"] == 0
    assert empty_positive["recall"] == empty_positive["f1"] == 0
    print("E3: OK")
    print("tau=0.5:", report)
    print("tau=0.8:", high)

TODO E3


**Réponse :** _Écrire votre explication ici._

## E4 · Entraîner, choisir un seuil, puis tester
Les données synthétiques contiennent **200 cas d’entraînement, 100 de validation et 100 de test**, avec une distance et une classe binaire. Elles simulent la probabilité qu’une marche dépasse 30 minutes ; ce ne sont pas des mesures de personnes réelles. Exécuter la cellule de données.

Ajuster les paramètres sur l’**entraînement** avec E2. Implémenter `choose_threshold` avec E3 : maximiser **F1 de validation** parmi `[0.2, 0.35, 0.5, 0.65, 0.8]`. En cas d’égalité, choisir le **seuil le plus élevé**. Renvoyer un seuil numérique.

Ne pas utiliser les classes de test pour ajuster les paramètres ou sélectionner le seuil.

In [8]:
def make_walk_splits():
    rng = np.random.default_rng(2026)
    splits = {}
    for name, size in [("train", 200), ("validation", 100), ("test", 100)]:
        x = rng.uniform(0.5, 6.5, size)
        event_probability = sigmoid(1.2 * (x - 3.5))
        c = (rng.random(size) < event_probability).astype(int)
        splits[name] = (x, c)
    return splits

walks = make_walk_splits()
x_train, c_train = walks["train"]
x_val, c_val = walks["validation"]
print("Training:", len(x_train), "| Validation:", len(x_val))

Training: 200 | Validation: 100


In [9]:
def choose_threshold(y_val, p_val, candidates):
    # TODO: evaluate F1 on VALIDATION for every candidate.
    # TODO: return the candidate with highest F1.
    # If several candidates tie, choose the HIGHEST threshold.
    return None

In [10]:
candidates = np.array([.2, .35, .5, .65, .8])
frozen_model = None
trained = fit_logistic(x_train, c_train)
if trained is None or classification_report_manual(c_val, np.full(len(c_val), .5), .5) is None:
    print("TODO E2 / E3")
else:
    p_val = sigmoid(trained["w"] * x_val + trained["b"])
    tau = choose_threshold(c_val, p_val, candidates)
    if tau is None:
        print("TODO E4")
    else:
        assert tau in candidates
        validation_rows = []
        print("tau    validation F1")
        for candidate in candidates:
            value = classification_report_manual(c_val, p_val, candidate)["f1"]
            validation_rows.append((float(candidate), value))
            print(f"{candidate:.2f}   {value:.6f}")
        best_f1 = max(value for candidate, value in validation_rows)
        assert np.isclose(classification_report_manual(c_val, p_val, tau)["f1"], best_f1)
        assert tau == max(t for t, value in validation_rows if value == best_f1)
        assert choose_threshold(np.array([1, 0]), np.array([.9, .1]),
                                np.array([.2, .8])) == .8
        frozen_model = (float(trained["w"]), float(trained["b"]), float(tau))
        print("Frozen (w, b, tau) =", frozen_model)

TODO E2 / E3


### Évaluation finale
Une fois `(w, b, tau)` figé, exécuter le test et rapporter les résultats. Ne pas réajuster après leur lecture.

Expliquer le rôle de chaque partie. Maximiser F1 de validation garantit-il de maximiser F1 de test ?

In [11]:
if frozen_model is None:
    print("TODO E4: freeze the model using validation first.")
else:
    w_final, b_final, tau_final = frozen_model
    x_test, c_test = walks["test"]
    p_test = sigmoid(w_final * x_test + b_final)
    test_report = classification_report_manual(c_test, p_test, tau_final)
    print("TEST:", test_report)
    print("TEST ROC-AUC:", roc_auc_score(c_test, p_test))
    print("TEST AP:", average_precision_score(c_test, p_test))
    # Verify the manual report with an independent library.
    test_labels = (p_test >= tau_final).astype(int)
    assert np.isclose(test_report["f1"], f1_score(c_test, test_labels, zero_division=0))

TODO E4: freeze the model using validation first.


**Réponse :** _Écrire votre explication ici._

## B1 · Expérience sur le pas
Réutiliser E1. Partir de `(10,0)` et effectuer 30 pas pour `0.01, 0.1, 0.4`. Tracer les trois pertes avec un axe vertical logarithmique. Garder les mêmes données et le même départ. Expliquer lenteur et divergence.

In [12]:
# TODO B1: compare eta = 0.01, 0.1, 0.4 on the linear regression data.
# Use the same start (10, 0) and 30 updates. Plot the loss trajectories.

## B2 · Reconstruire ROC sans `roc_curve`
Implémenter `roc_manual(y_true, scores)` ; les deux classes sont présentes. Partir au-dessus des scores, puis parcourir les **scores distincts décroissants**. Regrouper les égalités. Renvoyer `(fpr,tpr)` avec `(0,0)` et `(1,1)`.

Calculer AUC par trapèzes et comparer à 0.875. Une paire positif–négatif ex æquo contribue une demi-victoire. Ne pas intégrer PR par trapèzes pour calculer AP.

In [13]:
def roc_manual(y_true, scores):
    # TODO: group equal scores at the same threshold.
    # Return arrays (fpr, tpr), starting at (0,0) and ending at (1,1).
    # Inputs contain both classes.
    return None

In [14]:
curve = roc_manual(truth, scores)
if curve is None:
    print("TODO B2")
else:
    fpr, tpr = curve
    # Trapezoids, written explicitly for compatibility with NumPy versions.
    auc_manual = np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)
    assert np.isclose(auc_manual, .875)
    tied = roc_manual(np.array([0, 1]), np.array([.5, .5]))
    tfpr, ttpr = tied
    assert np.isclose(np.sum(np.diff(tfpr) * (ttpr[1:] + ttpr[:-1]) / 2), .5)
    print("B2: OK | AUC =", auc_manual)

TODO B2


## B3 · Un classifieur apparemment performant
Construire 1 000 classes : 10 positives et 990 négatives. Attribuer la probabilité 0 partout et utiliser le seuil 0.5. Calculer exactitude, rappel et F1 avec E3. Ce modèle est-il utile pour détecter les positifs ?

In [15]:
# TODO B3: 10 positives, 990 negatives; predict 0 for every case.
# Calculate accuracy, recall and F1, then interpret the result.

## Travail à rendre
Enregistrer sous `Nom_Prenom_TP02.ipynb` avec E1–E4, sorties, réponses et approfondissements réalisés. Redémarrer le noyau/l’environnement et tout exécuter avant de rendre. Conserver le seuil choisi et le bilan de test.

Les vérifications ne remplacent pas l’explication de l’algorithme et du protocole.

## Références
- [Stanford CS229 : apprentissage supervisé](https://see.stanford.edu/materials/aimlcs229/cs229-notes1.pdf)
- [scikit-learn : matrice de confusion](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [scikit-learn : précision–rappel](https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html)